# CubicasA5K — YOLOv8 Floor Plan Detection (Kaggle T4×2)
**Classes:** door · wall · window  
**Dataset:** 4178 train / 400 val / 400 test  

### Before running:
1. Add dataset via **+ Add Input** (upload cubicasa5k.zip or floor-plan dataset)
2. Settings → Accelerator → **GPU T4 x2**
3. Settings → Internet → **ON**
4. Run All


In [ ]:
# ── Cell 1: Check GPUs ────────────────────────────────────────────────────────
import torch, subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'GPUs    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name}  ({vram:.1f} GB)')


In [ ]:
# ── Cell 2: Install ultralytics ───────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')


In [ ]:
# ── Cell 3: Locate dataset ────────────────────────────────────────────────────
import os, glob

existing = glob.glob('/kaggle/input/**/cubicasa5k-2-6', recursive=True)

if existing:
    DATASET_ROOT = existing[0]
    print(f'Dataset found at: {DATASET_ROOT}')
else:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        EXTRACT_TO = '/kaggle/working/datasets'
        os.makedirs(EXTRACT_TO, exist_ok=True)
        print(f'Unzipping {zips[0]} ...')
        os.system(f'unzip -q -o "{zips[0]}" -d "{EXTRACT_TO}"')
        DATASET_ROOT = f'{EXTRACT_TO}/cubicasa5k-2-6'
        print('Done.')
    else:
        raise FileNotFoundError('No dataset found! Add cubicasa5k via + Add Input.')

print(f'Train images : {len(os.listdir(os.path.join(DATASET_ROOT, "train", "images")))}')
print(f'Val images   : {len(os.listdir(os.path.join(DATASET_ROOT, "valid", "images")))}')


In [ ]:
# ── Cell 4: Write data.yaml ───────────────────────────────────────────────────
import yaml, os

DATA_YAML_PATH = '/kaggle/working/data.yaml'

data = {
    'path':  DATASET_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 3,
    'names': ['door', 'wall', 'window']
}

with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

os.system(f'cat {DATA_YAML_PATH}')
print(f'data.yaml written to {DATA_YAML_PATH}')


In [ ]:
# ── Cell 5: Model settings for T4×2 (2×15GB VRAM) ────────────────────────────
import torch

NUM_GPUS = torch.cuda.device_count()
DEVICE   = list(range(NUM_GPUS))  # [0, 1] for T4×2
MODEL    = 'yolov8m.pt'
BATCH    = 16    # 8 per GPU — T4 handles it at 1024px
IMGSZ    = 1024  # high res for thin wall detection

print(f'GPUs    : {NUM_GPUS}  →  device={DEVICE}')
print(f'Model   : {MODEL}')
print(f'Batch   : {BATCH}  ({BATCH // NUM_GPUS} per GPU)')
print(f'Imgsz   : {IMGSZ}')


In [ ]:
# ── Cell 6: TRAIN with auto-save every 10 epochs ─────────────────────────────
import shutil, os, glob
from ultralytics import YOLO

SAVE_DIR  = '/kaggle/working/trained_models'
WEIGHTS   = '/kaggle/working/runs/cubicasa_yolo/weights'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Auto-save callback ────────────────────────────────────────────────────────
def autosave(trainer):
    epoch = trainer.epoch + 1
    if epoch % 10 == 0 or epoch == 1:
        try:
            if os.path.exists(str(trainer.best)):
                shutil.copy(str(trainer.best), f'{SAVE_DIR}/best.pt')
            if os.path.exists(str(trainer.last)):
                shutil.copy(str(trainer.last), f'{SAVE_DIR}/last.pt')
                shutil.copy(str(trainer.last), f'{SAVE_DIR}/last_epoch{epoch}.pt')
            print(f'  ✅ [Epoch {epoch}] Auto-saved to {SAVE_DIR}')
        except Exception as e:
            print(f'  ⚠️  [Epoch {epoch}] Auto-save failed: {e}')

# ── Resume from previous checkpoint if available ──────────────────────────────
# To resume: add your previous Kaggle output as input dataset,
# then last.pt will be found automatically at /kaggle/input/**/last.pt
prev_last = glob.glob('/kaggle/input/**/last.pt', recursive=True)
local_last = f'{WEIGHTS}/last.pt'

if prev_last and not os.path.exists(local_last):
    print(f'Checkpoint found at {prev_last[0]} — resuming...')
    os.makedirs(WEIGHTS, exist_ok=True)
    shutil.copy(prev_last[0], local_last)
    RESUME = True
else:
    RESUME = False

print(f'Resume: {RESUME}')

# ── Train ─────────────────────────────────────────────────────────────────────
model = YOLO(local_last if RESUME else MODEL)
model.add_callback('on_train_epoch_end', autosave)

results = model.train(
    data             = DATA_YAML_PATH,
    epochs           = 80,
    imgsz            = IMGSZ,
    batch            = BATCH,
    device           = DEVICE,
    workers          = 4,
    cache            = 'disk',
    patience         = 20,
    resume           = RESUME,
    multi_scale      = True,
    cos_lr           = True,
    close_mosaic     = 15,
    optimizer        = 'AdamW',
    lr0              = 0.001,
    lrf              = 0.01,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
    weight_decay     = 0.0005,
    box              = 7.5,
    cls              = 0.5,
    dfl              = 1.5,
    iou              = 0.7,
    mosaic           = 1.0,
    mixup            = 0.1,
    copy_paste       = 0.15,
    degrees          = 10.0,
    translate        = 0.1,
    scale            = 0.6,
    shear            = 2.0,
    fliplr           = 0.5,
    flipud           = 0.25,
    hsv_h            = 0.015,
    hsv_s            = 0.7,
    hsv_v            = 0.4,
    perspective      = 0.0001,
    project          = '/kaggle/working/runs',
    name             = 'cubicasa_yolo',
    exist_ok         = True,
)


In [ ]:
# ── Cell 7: Validate best model ───────────────────────────────────────────────
from ultralytics import YOLO

best_model = YOLO('/kaggle/working/runs/cubicasa_yolo/weights/best.pt')

metrics = best_model.val(
    data   = DATA_YAML_PATH,
    imgsz  = IMGSZ,
    split  = 'val',
    iou    = 0.45,
    conf   = 0.001,
    device = 0,
)

print('\n========== RESULTS ==========')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')
print(f'mAP@50    : {metrics.box.map50:.4f}')
print(f'mAP@50-95 : {metrics.box.map:.4f}')
print('==============================')
for i, name in enumerate(['door', 'wall', 'window']):
    p  = metrics.box.p[i]
    r  = metrics.box.r[i]
    m  = metrics.box.ap50[i]
    f1 = 2 * p * r / (p + r + 1e-9)
    print(f'  {name:6s}  P={p:.3f}  R={r:.3f}  mAP50={m:.3f}  F1={f1:.3f}')


In [ ]:
# ── Cell 8: Show training curves ──────────────────────────────────────────────
from IPython.display import Image, display
import os

run_dir = '/kaggle/working/runs/cubicasa_yolo'
for img in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    path = os.path.join(run_dir, img)
    if os.path.exists(path):
        print(f'--- {img} ---')
        display(Image(path))


In [ ]:
# ── Cell 9: Output ────────────────────────────────────────────────────────────
# best.pt is auto-saved to Kaggle Output tab
# Download: Notebook → Output → runs/cubicasa_yolo/weights/best.pt
import os
best = '/kaggle/working/runs/cubicasa_yolo/weights/best.pt'
size = os.path.getsize(best) / 1e6
print(f'best.pt size: {size:.1f} MB')
print('Download from Kaggle Output tab on the right panel.')
